# Notebook 6. NF4

The schemes so far use a uniform grid: sixteen equally spaced levels for INT4. NF4, the 4-bit NormalFloat of QLoRA (Dettmers et al., NeurIPS 2023), uses sixteen levels placed at the quantiles of a standard normal distribution, on the grounds that trained weights are close to normal. This notebook checks that premise on a real layer, builds the codebook, measures the reconstruction error against uniform INT4 at matched granularity, and then compares the two on GPT-2 at the granularities Pare offers for each, where the result is less clear-cut.

Prerequisites are Notebooks 1 and 2.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Are weights normal?

A uniform grid is the right choice for values spread evenly across their range. The histogram below is the first attention projection of GPT-2, with a normal density of the same standard deviation.

In [ ]:
from transformers import AutoModelForCausalLM

gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", torch_dtype=torch.float32)
W_real = gpt2.transformer.h[0].attn.c_attn.weight.detach()
w_flat = W_real.flatten().numpy()
del gpt2

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(w_flat, bins=200, density=True, alpha=0.6, label="GPT-2 layer 0 c_attn")
xs = np.linspace(w_flat.min(), w_flat.max(), 400)
sigma = w_flat.std()
density = np.exp(-(xs**2) / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
ax.plot(xs, density, "r--", label=f"normal, sigma = {sigma:.3f}")
ax.set_xlabel("weight value")
ax.legend()
plt.tight_layout()
plt.show()
tail = (np.abs(w_flat) > 3 * sigma).mean()
print(f"{tail:.2%} of the weights lie beyond three standard deviations (0.27% for a normal)")

The bulk is close to normal; the tails are heavier than normal, which is where a quantile grid gives up the most resolution.

## 2. The codebook

NF4 divides the standard normal into sixteen bins of equal probability and uses the bin centres, rescaled to $[-1, 1]$, as levels. Zero is included exactly, since zero weights are common, which leaves an odd number of remaining levels: seven negative and eight positive. The table below is `pare.core.functional._NF4_TABLE`, which reproduces Table 1 of the QLoRA paper, next to the uniform INT4 grid on the same range.

In [ ]:
from pare.core.functional import _NF4_TABLE

table = _NF4_TABLE.clone()
uniform = torch.linspace(-1, 1, 16)
print("NF4 levels:", [round(v, 3) for v in table.tolist()])

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.scatter(table, [1] * 16, s=40, label="NF4")
ax.scatter(uniform, [0] * 16, s=40, marker="x", label="uniform INT4")
ax.set_yticks([0, 1])
ax.set_yticklabels(["uniform", "NF4"])
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-0.6, 1.6)
plt.tight_layout()
plt.show()

## 3. Quantizing with a codebook

With a non-uniform grid there is no formula for the code; each value is mapped to the index of its nearest level. Pare normalizes a weight row by its maximum magnitude so that it lies in $[-1, 1]$, looks up the nearest level, and stores the 4-bit index with one scale per row. The functions below are the ones in `pare.core.functional`, and the cell checks that they agree.

In [ ]:
from pare.core.functional import dequantize_nf4 as pare_dequantize_nf4
from pare.core.functional import quantize_nf4 as pare_quantize_nf4


def quantize_nf4(W, scale):
    # W: [n_out, n_in]; scale: [n_out, 1] per-row absolute maximum. Returns indices in [0, 15].
    distances = (W / scale.clamp(min=1e-8)).unsqueeze(-1).sub(table).abs()
    return distances.argmin(dim=-1).to(torch.int32)


def dequantize_nf4(indices, scale):
    return table[indices.long()] * scale


torch.manual_seed(0)
W = torch.randn(64, 256) * 0.02
scale = W.abs().amax(dim=1, keepdim=True)
idx = quantize_nf4(W, scale)
assert torch.equal(idx, pare_quantize_nf4(W, scale))
assert torch.allclose(dequantize_nf4(idx, scale), pare_dequantize_nf4(idx, scale))
print("quantize_nf4 and dequantize_nf4 match pare.core.functional")
print(f"indices in use: {idx.unique().tolist()}")
print(f"reconstruction MAE {(dequantize_nf4(idx, scale) - W).abs().mean():.6f}")

## 4. NF4 against uniform INT4 at the same granularity

Same weights, same four bits, one scale per row for both; only the grid differs. This isolates what the quantile placement buys.

In [ ]:
from pare.core.dtype import QuantDtype
from pare.core.functional import dequantize_tensor, quantize_tensor
from pare.core.scale import compute_scale


def mse(a, b):
    return (a - b).pow(2).mean().item()


torch.manual_seed(42)
W = torch.randn(64, 128)
scale = W.abs().amax(dim=1, keepdim=True)
W_nf4 = dequantize_nf4(quantize_nf4(W, scale), scale)

s, z = compute_scale(W, QuantDtype.INT4, granularity="per_channel")
W_int4 = dequantize_tensor(quantize_tensor(W, s, z, QuantDtype.INT4), s, z).reshape_as(W)

print(f"NF4 per row:          MSE {mse(W, W_nf4):.6f}")
print(f"uniform INT4 per row: MSE {mse(W, W_int4):.6f}")
print(f"ratio {mse(W, W_int4) / mse(W, W_nf4):.2f}")

# The same comparison on the real GPT-2 weights of Section 1.
scale_real = W_real.abs().amax(dim=1, keepdim=True)
W_real_nf4 = dequantize_nf4(quantize_nf4(W_real, scale_real), scale_real)
s, z = compute_scale(W_real, QuantDtype.INT4, granularity="per_channel")
W_real_int4 = dequantize_tensor(quantize_tensor(W_real, s, z, QuantDtype.INT4), s, z)
W_real_int4 = W_real_int4.reshape_as(W_real)
print(f"GPT-2 c_attn, NF4 per row MSE {mse(W_real, W_real_nf4):.3e}, "
      f"uniform INT4 per row MSE {mse(W_real, W_real_int4):.3e}, "
      f"ratio {mse(W_real, W_real_int4) / mse(W_real, W_real_nf4):.2f}")

At matched granularity NF4 has the lower error on both the Gaussian matrix and the real layer, and Pare's test suite checks the Gaussian case. The margin is larger on the real layer. Its heavy tails set a wide row maximum, so the uniform grid spends most of its sixteen levels on a range that few weights use, while NF4 keeps its levels concentrated where the weights are.

## 5. Storage and where NF4 is used

NF4 and uniform INT4 both store four bits per weight plus their scales, so their memory is the same at the same granularity. NF4 was designed for QLoRA, where a frozen base model is stored in NF4 and small LoRA adapters are trained on top; during training the NF4 weights are dequantized to the compute dtype inside each matmul and never stored in full precision. Pare's NF4 keeps one scale per row; the bitsandbytes implementation used by QLoRA quantizes blocks of 64 weights and can quantize the scales themselves, so the two formats are not interchangeable on disk. For serving, INT4 checkpoints with group scales (GPTQ, AWQ) are what the fused kernels target.

## 6. NF4 and INT4 in Pare on GPT-2

Pare's NF4 is always per row, while its INT4 path supports groups. The cell compares FP32, RTN INT4 with groups of 128 and NF4 on GPT-2, with 20 evaluation windows of 512 tokens. This comparison is not matched: it pits a finer granularity with a uniform grid against a coarser granularity with a better grid, which is the choice a Pare user faces.

In [ ]:
from transformers import AutoTokenizer

from pare import QuantConfig, quantize
from pare.eval.perplexity import evaluate_perplexity

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)


def fresh_model():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)


def ppl(model):
    return evaluate_perplexity(
        model, tokenizer, dataset="wikitext2", seq_len=512, n_samples=20, device="cpu"
    )


results = {"FP32": ppl(fresh_model())}
configs = {
    "RTN INT4 g=128": QuantConfig(bits=4, scheme="rtn", granularity="per_group", group_size=128),
    "RTN INT4 per row": QuantConfig(bits=4, scheme="rtn", granularity="per_channel"),
    "NF4 per row": QuantConfig(bits=4, dtype=QuantDtype.NF4, scheme="rtn"),
}
for label, cfg in configs.items():
    model = fresh_model()
    quantize(model, cfg)
    results[label] = ppl(model)

for label, value in results.items():
    print(f"{label:>18}: PPL {value:6.2f}  ({value - results['FP32']:+.2f})")

Read the three quantized rows together. Against INT4 at the same per-row granularity, NF4 should be ahead, as in Section 4. Against INT4 with groups of 128, the finer granularity can outweigh the better grid, because a group of 128 adapts its step to local magnitudes while one scale per row of 768 or 3072 weights cannot. The grid and the granularity both matter, and in Pare they are not available in every combination.

## Exercises

1. Build a 16-level codebook from the quantiles of a Laplace distribution instead of a normal one and repeat Section 4 on the GPT-2 weights, whose tails are heavier than normal. Which codebook has the lower error?
2. Implement NF4 with one scale per group of 64 weights, as bitsandbytes does, and add it to the Section 4 comparison against INT4 with groups of 64.
3. In Section 6, replace GPT-2 by `Qwen/Qwen2.5-0.5B`. Does the ordering of the three quantized rows change?

## Summary

| | Uniform INT4 | NF4 |
|---|---|---|
| Levels | 16, equally spaced | 16, quantiles of the standard normal |
| Code | formula, $\lfloor w/s \rceil + z$ | nearest-level lookup |
| Granularity in Pare | per tensor, per row or per group | per row |
| Lower error at matched granularity | | yes, on Gaussian and GPT-2 weights |
| Designed for | serving kernels | QLoRA fine-tuning |

Notebook 7 turns to FP8, a floating-point grid whose spacing follows the magnitude of each value rather than a fixed codebook.